# Model Evaluation
## Telco Customer Churn Prediction

This notebook provides comprehensive evaluation of the trained churn prediction model.

### Contents:
1. Load Models and Data
2. Classification Metrics
3. ROC and PR Curves
4. Confusion Matrix Analysis
5. Threshold Optimization
6. Business Impact Analysis
7. Final Report

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings

# Add src to path
sys.path.append('../src')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, classification_report, average_precision_score
)
import joblib

warnings.filterwarnings('ignore')

# Import project modules
from config import (
    TEST_DATA_DIR, MODELS_DIR, FIGURES_DIR, create_directories
)
from models.classification import ChurnClassifier
from evaluation.metrics import (
    evaluate_classification, plot_confusion_matrix, plot_roc_curve,
    plot_precision_recall_curve, plot_threshold_analysis,
    find_optimal_threshold
)

create_directories()
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries imported!')

## 1. Load Models and Data

In [ ]:
# Load test data
test = pd.read_csv(TEST_DATA_DIR / 'test.csv')
print(f'Test set: {len(test)} samples')
print(f'Churn rate: {test["Churn_Binary"].mean():.2%}')

In [ ]:
# Load trained model
model = ChurnClassifier.load(MODELS_DIR / 'churn_model.pkl')
print(f'Model type: {model.model_type}')

# Load feature info
feature_info = joblib.load(MODELS_DIR / 'feature_info.pkl')
feature_cols = feature_info['feature_columns']
print(f'Number of features: {len(feature_cols)}')

In [ ]:
# Prepare test data
X_test = test[feature_cols].values
y_test = test['Churn_Binary'].values

# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Predictions made for {len(y_test)} samples')

## 2. Classification Metrics

In [ ]:
# Comprehensive evaluation
metrics = evaluate_classification(y_test, y_pred, y_proba)

In [ ]:
# Metrics visualization
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'PR-AUC']
metric_values = [
    metrics['accuracy'], metrics['precision'], metrics['recall'],
    metrics['f1'], metrics['roc_auc'], metrics['pr_auc']
]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
bars = ax.barh(metric_names, metric_values, color=colors)
ax.set_xlabel('Score')
ax.set_title('Classification Performance Metrics')
ax.set_xlim([0, 1])

# Add value labels
for bar, value in zip(bars, metric_values):
    ax.text(value + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{value:.3f}', va='center', fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'metrics_summary.png', dpi=150)
plt.show()

## 3. ROC and PR Curves

In [ ]:
# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

axes[1].plot(recall, precision, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.3f})')
axes[1].fill_between(recall, precision, alpha=0.2, color='green')
baseline = y_test.mean()
axes[1].axhline(y=baseline, color='navy', linestyle='--', label=f'Baseline ({baseline:.3f})')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_pr_curves.png', dpi=150)
plt.show()

## 4. Confusion Matrix Analysis

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)')

# Percentages
cm_pct = cm / cm.sum() * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Percentages)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

print('\nConfusion Matrix Breakdown:')
print(f'True Negatives (Correctly predicted No Churn): {tn}')
print(f'False Positives (Incorrectly predicted Churn): {fp}')
print(f'False Negatives (Missed Churners): {fn}')
print(f'True Positives (Correctly predicted Churn): {tp}')

## 5. Threshold Optimization

In [ ]:
# Analyze metrics at different thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_metrics = {'threshold': [], 'precision': [], 'recall': [], 'f1': [], 'accuracy': []}

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    threshold_metrics['threshold'].append(thresh)
    threshold_metrics['precision'].append(precision_score(y_test, y_pred_thresh, zero_division=0))
    threshold_metrics['recall'].append(recall_score(y_test, y_pred_thresh, zero_division=0))
    threshold_metrics['f1'].append(f1_score(y_test, y_pred_thresh, zero_division=0))
    threshold_metrics['accuracy'].append(accuracy_score(y_test, y_pred_thresh))

threshold_df = pd.DataFrame(threshold_metrics)

In [ ]:
# Plot threshold analysis
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(threshold_df['threshold'], threshold_df['precision'], 'b-', label='Precision', lw=2)
ax.plot(threshold_df['threshold'], threshold_df['recall'], 'g-', label='Recall', lw=2)
ax.plot(threshold_df['threshold'], threshold_df['f1'], 'r-', label='F1 Score', lw=2)
ax.plot(threshold_df['threshold'], threshold_df['accuracy'], 'm--', label='Accuracy', lw=2)

# Find optimal threshold for F1
best_idx = threshold_df['f1'].idxmax()
best_thresh = threshold_df.loc[best_idx, 'threshold']
ax.axvline(x=best_thresh, color='gray', linestyle='--', lw=2, label=f'Optimal (F1): {best_thresh:.2f}')

ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Metrics vs Classification Threshold')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_analysis.png', dpi=150)
plt.show()

print(f'\nOptimal threshold for F1 Score: {best_thresh:.2f}')
print(f'Metrics at optimal threshold:')
print(threshold_df.loc[best_idx])

In [ ]:
# Compare default vs optimal threshold
print('\nComparison: Default (0.5) vs Optimal Threshold\n')
print('-' * 50)

for thresh in [0.5, best_thresh]:
    y_pred_t = (y_proba >= thresh).astype(int)
    print(f'\nThreshold: {thresh:.2f}')
    print(f'  Accuracy:  {accuracy_score(y_test, y_pred_t):.4f}')
    print(f'  Precision: {precision_score(y_test, y_pred_t):.4f}')
    print(f'  Recall:    {recall_score(y_test, y_pred_t):.4f}')
    print(f'  F1 Score:  {f1_score(y_test, y_pred_t):.4f}')

## 6. Business Impact Analysis

In [ ]:
# Define business costs and values
print('Business Impact Analysis')
print('='*50)

# Assumptions
avg_monthly_revenue = test['MonthlyCharges'].mean()
avg_customer_lifetime = 24  # months
retention_cost = 50  # cost per retention effort
retention_success_rate = 0.50  # 50% of predicted churners can be retained

print(f'\nAssumptions:')
print(f'  Average Monthly Revenue: ${avg_monthly_revenue:.2f}')
print(f'  Average Customer Lifetime: {avg_customer_lifetime} months')
print(f'  Retention Campaign Cost: ${retention_cost}')
print(f'  Retention Success Rate: {retention_success_rate:.0%}')

In [ ]:
# Calculate business metrics
customer_value = avg_monthly_revenue * avg_customer_lifetime

# Without model (no intervention)
total_churners = y_test.sum()
revenue_lost_no_model = total_churners * customer_value

# With model
correctly_identified_churners = tp  # True Positives
saved_customers = correctly_identified_churners * retention_success_rate
revenue_saved = saved_customers * customer_value

# Costs
total_predicted_churners = (y_pred == 1).sum()
retention_campaign_cost = total_predicted_churners * retention_cost

# Net benefit
net_benefit = revenue_saved - retention_campaign_cost

print(f'\nBusiness Impact Calculation:')
print('-'*50)
print(f'Customer Lifetime Value: ${customer_value:.2f}')
print(f'\nWithout ML Model:')
print(f'  Total Churners: {total_churners}')
print(f'  Potential Revenue Lost: ${revenue_lost_no_model:,.2f}')
print(f'\nWith ML Model:')
print(f'  Predicted Churners: {total_predicted_churners}')
print(f'  Correctly Identified Churners: {tp}')
print(f'  Estimated Saved Customers: {saved_customers:.0f}')
print(f'  Revenue Saved: ${revenue_saved:,.2f}')
print(f'  Retention Campaign Cost: ${retention_campaign_cost:,.2f}')
print(f'\n  NET BENEFIT: ${net_benefit:,.2f}')

In [ ]:
# ROI Analysis
roi = (net_benefit / retention_campaign_cost) * 100 if retention_campaign_cost > 0 else 0

print(f'\nROI Analysis:')
print(f'  Investment (Retention Campaign): ${retention_campaign_cost:,.2f}')
print(f'  Return (Revenue Saved): ${revenue_saved:,.2f}')
print(f'  Net Return: ${net_benefit:,.2f}')
print(f'  ROI: {roi:.1f}%')

In [ ]:
# Cost-benefit at different thresholds
threshold_business = []

for thresh in np.arange(0.2, 0.8, 0.1):
    y_pred_t = (y_proba >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    
    predicted_churners = (y_pred_t == 1).sum()
    saved = tp_t * retention_success_rate
    revenue = saved * customer_value
    cost = predicted_churners * retention_cost
    net = revenue - cost
    
    threshold_business.append({
        'Threshold': thresh,
        'Predicted Churners': predicted_churners,
        'True Positives': tp_t,
        'Revenue Saved': revenue,
        'Campaign Cost': cost,
        'Net Benefit': net
    })

business_df = pd.DataFrame(threshold_business)
print('\nBusiness Impact at Different Thresholds:')
print(business_df.to_string(index=False))

In [ ]:
# Visualize business impact
fig, ax = plt.subplots(figsize=(10, 6))

x = business_df['Threshold']
ax.plot(x, business_df['Revenue Saved']/1000, 'g-o', label='Revenue Saved ($K)', lw=2)
ax.plot(x, business_df['Campaign Cost']/1000, 'r-s', label='Campaign Cost ($K)', lw=2)
ax.plot(x, business_df['Net Benefit']/1000, 'b-^', label='Net Benefit ($K)', lw=2, markersize=10)

# Find optimal business threshold
best_business_idx = business_df['Net Benefit'].idxmax()
best_business_thresh = business_df.loc[best_business_idx, 'Threshold']
ax.axvline(x=best_business_thresh, color='gray', linestyle='--', label=f'Optimal: {best_business_thresh:.1f}')

ax.set_xlabel('Threshold')
ax.set_ylabel('Amount ($K)')
ax.set_title('Business Impact vs Classification Threshold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'business_impact.png', dpi=150)
plt.show()

## 7. Final Report

In [ ]:
# Generate final report
print('='*70)
print('FINAL MODEL EVALUATION REPORT')
print('='*70)

print(f'''
MODEL INFORMATION
-----------------
Model Type: {model.model_type}
Test Set Size: {len(y_test)} customers
Churn Rate in Test Set: {y_test.mean():.2%}

CLASSIFICATION PERFORMANCE
--------------------------
Accuracy:     {metrics['accuracy']:.4f}
Precision:    {metrics['precision']:.4f}
Recall:       {metrics['recall']:.4f}
F1 Score:     {metrics['f1']:.4f}
ROC-AUC:      {metrics['roc_auc']:.4f}
PR-AUC:       {metrics['pr_auc']:.4f}

CONFUSION MATRIX
----------------
True Negatives:  {tn} (correct non-churn predictions)
False Positives: {fp} (false alarms)
False Negatives: {fn} (missed churners)
True Positives:  {tp} (correctly identified churners)

THRESHOLD ANALYSIS
------------------
Default Threshold: 0.50
Optimal Threshold (F1): {best_thresh:.2f}
Optimal Threshold (Business): {best_business_thresh:.2f}

BUSINESS IMPACT (at default threshold)
--------------------------------------
Customer Lifetime Value: ${customer_value:,.2f}
Predicted Churners: {total_predicted_churners}
Estimated Customers Saved: {saved_customers:.0f}
Revenue Saved: ${revenue_saved:,.2f}
Retention Campaign Cost: ${retention_campaign_cost:,.2f}
Net Benefit: ${net_benefit:,.2f}
ROI: {roi:.1f}%

KEY INSIGHTS
------------
1. The model achieves {metrics['roc_auc']:.1%} ROC-AUC, indicating strong 
   discriminative ability between churners and non-churners.

2. With {metrics['recall']:.1%} recall, we can identify {tp} out of {y_test.sum()} 
   actual churners in the test set.

3. The expected ROI of {roi:.0f}% demonstrates significant business value.

4. Lower thresholds increase recall but also false positives (higher cost).
   The optimal business threshold depends on retention campaign costs.

RECOMMENDATIONS
---------------
1. Deploy model with threshold around {best_business_thresh:.2f} for optimal ROI
2. Focus retention efforts on high-probability churners first
3. Monitor model performance monthly and retrain quarterly
4. Collect feedback on retention campaign effectiveness
''')

print('='*70)

In [ ]:
# Save evaluation results
evaluation_results = {
    'metrics': metrics,
    'threshold_analysis': threshold_df.to_dict(),
    'business_impact': business_df.to_dict(),
    'optimal_threshold_f1': best_thresh,
    'optimal_threshold_business': best_business_thresh
}

joblib.dump(evaluation_results, MODELS_DIR / 'evaluation_results.pkl')
print(f'\nEvaluation results saved to {MODELS_DIR / "evaluation_results.pkl"}')
print(f'Figures saved to {FIGURES_DIR}/')